## 1. Імпорти моделей з src.models.*


In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.pipelines.wfv_orchestrator import WFVConfig, run_wfv, save_wfv_results
from src.pipelines.evaluation_pipeline import evaluate_experiment_a
from src.models.statistical import RandomWalkModel, DLinearModel, ARIMAGARCHModel
from src.models.ml_models import XGBoostForecaster, LightGBMForecaster
from src.models.nbeats_model import NBEATSForecaster
from src.models.patchtst_model import PatchTSTForecaster
from src.models.tft_model import TFTForecaster


## 2. Запуск WFV для кожної моделі


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

project_root = Path('..').resolve()
processed_dir = project_root / 'data' / 'processed'
X = pd.read_parquet(processed_dir / 'feature_matrix.parquet')
y = pd.read_parquet(processed_dir / 'target_h1.parquet')
y_series = y.iloc[:, 0]
config = WFVConfig(horizon=1)  # міняємо тільки тут

models = [
    ('random_walk', RandomWalkModel()),
    ('dlinear', DLinearModel()),
    ('arima_garch', ARIMAGARCHModel()),
    ('xgboost', XGBoostForecaster()),
    ('lightgbm', LightGBMForecaster()),
    ('nbeats', NBEATSForecaster(**config.model_kwargs)),
    ('patchtst', PatchTSTForecaster(**config.model_kwargs)),
    ('tft', TFTForecaster(**config.model_kwargs)),
]

results = {}
for name, model in models:
    try:
        iterations, preds = run_wfv(X, y_series, model, config)
        results[name] = (iterations, preds)
    except Exception as exc:
        print(f'WFV failed for {name}: {exc}')


FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/feature_matrix.parquet'

## 3. Збереження predictions через save_wfv_results


In [ ]:
for name, (iterations, preds) in results.items():
    save_wfv_results(iterations, preds, model_name=name, output_dir=processed_dir)


## 4. Виклик evaluate_experiment_a → таблиця метрик


In [ ]:
from src.pipelines.evaluation_pipeline import load_all_predictions

preds = load_all_predictions(processed_dir)
test_df = pd.read_parquet(processed_dir / 'test_returns.parquet')
y_test = test_df['brent_return']
train_df = pd.read_parquet(processed_dir / 'train_returns.parquet')
train_window = train_df['brent_return'].tail(config.w_train)

results_a = evaluate_experiment_a(preds, y_test, train_window)
results_a


## 5. Візуалізація: predicted vs actual для brent h=1 (останні 200 днів)


In [ ]:
window = 200
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test.index[-window:], y=y_test.values[-window:], name='actual'))
for name, series in preds.items():
    fig.add_trace(go.Scatter(x=series.index[-window:], y=series.values[-window:], name=name))
fig.update_layout(title='Predicted vs Actual (last 200 days)')
fig.show()


## 6. Prediction intervals plot для ARIMA-GARCH та TFT


In [ ]:
interval_models = {
    'arima_garch': ARIMAGARCHModel(),
    'tft': TFTForecaster(**config.model_kwargs),
}
fig = go.Figure()
fig.add_trace(go.Scatter(x=y_test.index[-window:], y=y_test.values[-window:], name='actual'))
for name, model in interval_models.items():
    try:
        model.fit(X, y_series)
        lower, upper = model.predict_interval(X.tail(window))
        fig.add_trace(go.Scatter(x=X.index[-window:], y=lower, name=f'{name}_lower'))
        fig.add_trace(go.Scatter(x=X.index[-window:], y=upper, name=f'{name}_upper'))
    except Exception as exc:
        print(f'Interval failed for {name}: {exc}')
fig.update_layout(title='Prediction Intervals (last 200 days)')
fig.show()


## 7. DM test heatmap: матриця p-values для всіх пар моделей


In [ ]:
from src.models.base import diebold_mariano_test

model_names = list(preds.keys())
pvals = np.zeros((len(model_names), len(model_names)))
for i, mi in enumerate(model_names):
    for j, mj in enumerate(model_names):
        if i == j:
            pvals[i, j] = 1.0
            continue
        err_i = y_test.loc[preds[mi].index] - preds[mi]
        err_j = y_test.loc[preds[mj].index] - preds[mj]
        dm = diebold_mariano_test(err_i, err_j, h=1)
        pvals[i, j] = dm['p_value']

fig = px.imshow(pvals, x=model_names, y=model_names, text_auto=True, title='DM test p-values')
fig.show()


## 8. Rolling MASE plot (Ukraine war 2022 focus)


In [ ]:
from src.models.base import calculate_window_mase

rolling_window = 100
fig = go.Figure()
train_window = train_df['brent_return'].tail(config.w_train)
for name, series in preds.items():
    aligned_y = y_test.loc[series.index]
    mase_series = []
    for start in range(0, len(series) - rolling_window + 1):
        end = start + rolling_window
        mase = calculate_window_mase(
            train_window,
            aligned_y.iloc[start:end],
            series.iloc[start:end],
        )
        mase_series.append(mase)
    idx = series.index[rolling_window - 1:]
    fig.add_trace(go.Scatter(x=idx, y=mase_series, name=name))
fig.update_layout(title='Rolling MASE')
fig.show()
